# LangChain: Agents

## Outline:

* Using built in LangChain tools: DuckDuckGo search and Wikipedia
* Defining your own tools

Agent（智能体）跟前面的 Chain 不同：Chain 的执行步骤是写死的，而 Agent 会让 LLM 自己"思考-决定该用哪个工具-执行-观察结果-再思考"，
循环推理直到得出最终答案（也就是常说的 ReAct 模式）。本节演示如何给 Agent 装配内置工具（数学计算器、维基百科搜索、Python 解释器）
以及如何自己写一个自定义工具（获取今天日期）。

> 注：本 notebook 用到的 `load_tools(["llm-math", ...])` 依赖 `numexpr` 包、`langchain_community.agent_toolkits.load_tools`
> 依赖 `mypy_extensions` 包，这两个是本环境原先缺失的**间接依赖**（不是代码 bug），已通过
> `pip install numexpr mypy_extensions` 补装到 venv 里，否则 `load_tools(["llm-math", ...])` 会直接报
> `ModuleNotFoundError`。这属于环境依赖缺失，不属于本 notebook 代码逻辑问题。

In [ ]:
# 加载 .env 中的环境变量（如 OPENAI_API_KEY）
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

# 屏蔽掉不影响运行结果的警告信息，让输出更干净
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# account for deprecation of LLM model
# 根据当前日期判断该用哪个 gpt-3.5-turbo 版本名，逻辑本身没问题
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
# 【版本兼容性修复】原代码全部从 langchain.xxx 导入，这些路径在当前 langchain 1.4.0 中已不存在：
#   create_python_agent, PythonREPLTool, PythonREPL -> 属于实验性功能，在 langchain_experimental 包里
#   load_tools, initialize_agent, AgentType          -> 经典 agent 组件，在 langchain_classic.agents 里
#   ChatOpenAI                                        -> langchain_openai
from langchain_experimental.agents.agent_toolkits import create_python_agent
from langchain_classic.agents import load_tools, initialize_agent
from langchain_classic.agents import AgentType
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_experimental.utilities import PythonREPL
from langchain_openai import ChatOpenAI

In [ ]:
# temperature=0：Agent 的推理过程需要稳定、可预测，不需要创意
llm = ChatOpenAI(temperature=0, model=llm_model)

In [ ]:
# load_tools 是 LangChain 提供的"内置工具库"加载器：
#   "llm-math"  -> 基于 LLM + 数值计算库(numexpr) 的计算器工具
#   "wikipedia" -> 维基百科搜索工具，需要 wikipedia 这个 Python 包
tools = load_tools(["llm-math","wikipedia"], llm=llm)

In [ ]:
# initialize_agent 把 tools（可选工具集）和 llm（推理大脑）组装成一个 Agent
# TODO: 请在此处补全代码
# 提示：agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, handle_parsing_errors=True, verbose=True
agent = None

In [ ]:
# 【提示】agent(...) 是旧式的"把 agent 当函数调用"写法，当前版本仍可用（deprecated），
# 等价于 agent.invoke("What is the 25% of 300?")
# 预期：Agent 会识别出这是数学问题，调用 llm-math 工具计算
agent("What is the 25% of 300?")

In [ ]:
# 预期：Agent 会识别出这是需要查资料的问题，调用 wikipedia 工具搜索 Tom M. Mitchell 相关信息
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
result = agent(question)

In [ ]:
# create_python_agent：专门用来执行 Python 代码的 Agent，内部工具是一个 Python REPL
# TODO: 请在此处补全代码
agent = None

In [ ]:
# 一份 [名, 姓] 的客户名单，之后让 Agent 写代码对它排序
customer_list = [["Harrison", "Chase"],
                 ["Lang", "Chain"],
                 ["Dolly", "Too"],
                 ["Elle", "Elem"],
                 ["Geoff","Fusion"],
                 ["Trance","Former"],
                 ["Jen","Ayai"]
                ]

In [ ]:
# 【提示】agent.run(...) 是旧式调用方法，当前版本仍可用（deprecated），新写法是 agent.invoke({"input": ...})
# 预期：Agent 会自己写一段 Python 排序代码，用 PythonREPLTool 执行，再把结果打印出来
agent.run(f"""Sort these customers by \
last name and then first name \
and print the output: {customer_list}""")

In [ ]:
# 提示：新版要用 langchain_core.globals.set_debug(True/False)，而不是 langchain.debug = True/False
# TODO: 请在此处补全代码（打开 debug，跑一遍 agent.run(...)，再关闭 debug）


In [ ]:
# 【版本兼容性修复】原代码 from langchain.agents import tool 已不存在（会 ImportError），
# @tool 装饰器现在的标准位置是 langchain_core.tools（也可以从 langchain_classic.agents 导入，效果一样）
# @tool 装饰器可以把任意 Python 函数包装成一个 Agent 可调用的"自定义工具"，
# 函数的 docstring 会被当成这个工具的说明，供 LLM 判断"什么时候该用这个工具"
from langchain_core.tools import tool
from datetime import date

In [ ]:
# 自定义工具：返回今天的日期
# TODO: 请在此处补全代码
# 提示：用 @tool 装饰一个函数 time(text: str) -> str，docstring 说明用途，返回 str(date.today())


In [ ]:
# 把自定义的 time 工具加入原有工具列表，重新初始化一个 Agent
# TODO: 请在此处补全代码
agent = None

In [ ]:
# 用 try/except 兜底：如果模型没有正确调用 time 工具，或者过程中出现解析错误，就打印一条提示而不是让 notebook 中断
# （这是原课程为了在直播/录制环境中演示稳定性而设计的写法，这里保留原样）
try:
    result = agent("whats the date today?")
except:
    print("exception on external access")